# Mnemo on Kaggle: the whole system + its evaluation (version 3)

**Mnemo** = a chest-camera memory assistant. Video → a short text log of what you did (no video is kept) →
answers to *"did I close the fridge?"*, with the time.

What is ours and what this notebook measures:
1. **Interaction trigger**: our HOI-TCN model decides *when* the big vision-language model (Qwen3-VL) looks.
   Compared against "every N-th window" with the same number of calls.
2. **Grounded answers**: only from the log, with times, or *"I didn't see that"*. Compared against asking the big model directly.
3. **Long-term memory**: questions about a whole morning (Experiment C), where a single model call cannot keep up.
4. Object-state alerts (*"the oven was last seen OPEN at 9:00"*), checked by asking the model again.

**Settings (right panel):** Accelerator **GPU T4** · Internet **On** · Input: dataset **mnemofyp** (latest version).

**How to run (no risk of losing results):** top right **Save Version** → **Save & Run All (Commit)** → Save.
Kaggle runs everything on its own (about 2 hours) even if you close the browser. When the version is finished,
open it → **Output** → download `mnemo_results.zip`.

## 1. Code + packages (~3 min)

In [ ]:
import os, shutil, glob, sys, subprocess
WORK = '/kaggle/working'
CODE = f'{WORK}/mnemo'
code_dir = code_zip = None
for root, dirs, files in os.walk('/kaggle/input', followlinks=True):
    if 'mnemo.py' in files and 'modules' in dirs:
        code_dir = root
    if 'mnemo_code.zip' in files:
        code_zip = os.path.join(root, 'mnemo_code.zip')
print('code folder:', code_dir, '| zip:', code_zip)
assert code_dir or code_zip, 'Code not found: add the mnemofyp dataset (Add Input) and wait until it is attached'
shutil.rmtree(CODE, ignore_errors=True)
if code_dir:
    shutil.copytree(code_dir, CODE)
else:
    os.makedirs(CODE)
    !cd "{CODE}" && unzip -q "{code_zip}"
%cd {CODE}
assert os.path.exists('evaluation/long_qa.py'), 'OLD code: upload the new mnemo_code.zip as a new version of mnemofyp'
!pip install -q -U "transformers>=4.57" accelerate mediapipe ultralytics
import torch, numpy, transformers
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE: switch Accelerator to GPU T4!')
print('transformers', transformers.__version__, '| numpy', numpy.__version__)

## 2. Start downloading the test videos (runs in the background)
20 random minutes of EPIC-KITCHENS from **people P02, P06, P08, P14, P26**, whom the trigger never saw in training.
The official annotations give the TRUE actions, so everything is scored automatically.
The download runs **while the big model loads** in step 3.

In [ ]:
import urllib.request
EK = f'{WORK}/epic'
CLIPS_DIR = f'{WORK}/clips_v2'
os.makedirs(EK, exist_ok=True); os.makedirs(CLIPS_DIR, exist_ok=True)
A = 'https://raw.githubusercontent.com/epic-kitchens/epic-kitchens-100-annotations/master'
D = 'https://raw.githubusercontent.com/epic-kitchens/epic-kitchens-download-scripts/master/data'
for url in [f'{A}/EPIC_100_train.csv', f'{A}/EPIC_100_verb_classes.csv', f'{A}/EPIC_100_noun_classes.csv',
            f'{A}/EPIC_100_video_info.csv', f'{D}/epic_55_splits.csv']:
    dst = f'{EK}/{url.rsplit("/", 1)[1]}'
    if not os.path.exists(dst):
        urllib.request.urlretrieve(url, dst)
ANN = f'{EK}/EPIC_100_train.csv'
N_CLIPS = 20
TEST_P = ['P02', 'P06', 'P08', 'P14', 'P26']
SEED = 7            # confirmation run: 20 clips never used before (seed 1 = the clips used while developing)
FETCH_LOG = f'{WORK}/fetch.log'
FETCH = subprocess.Popen([sys.executable, '-m', 'tools.fetch_epic_windows', '--data', EK, '--out', CLIPS_DIR,
                          '--select', 'random', '--windows', str(N_CLIPS), '--window-s', '60', '--workers', '8',
                          '--participants', *TEST_P, '--seed', str(SEED)],
                         cwd=CODE, stdout=open(FETCH_LOG, 'w'), stderr=subprocess.STDOUT)
print('download started in the background → continue with step 3')

## 3. Load the models (~5 min the first time: Qwen3-VL-4B is 9 GB)

In [ ]:
import time
import config as C
from modules.vlm_logger import VLMDescriber, ActionLogger
from modules.hoi_gate import HOIGate, gate_decisions
t = time.time()
describer = VLMDescriber(C.VLM_MODEL)            # stage 1 + stage 2 (same model)
gate = HOIGate('models/hoi_tcn.pt')               # our trigger: hands + objects + HOI-TCN
print(f'loaded in {time.time() - t:.0f}s | GPU memory {torch.cuda.memory_allocated() / 1e9:.1f} GB')
print(f'prompt version {C.VLM_PROMPT_VERSION} | trigger threshold {C.GATE_THRESHOLD}')

## 4. Wait for the downloads to finish

In [ ]:
FETCH.wait()
print(open(FETCH_LOG).read()[-1200:])
CLIPS = sorted(glob.glob(f'{CLIPS_DIR}/*_w*.mp4'))
print(len(CLIPS), 'clips:', [os.path.basename(c) for c in CLIPS])

## 5. LIVE DEMO: one clip through the whole system
**Show this to the professor.** Change `VIDEO` to any clip from the list above.
- `skipped` = the trigger saw no hand interaction, so the big model was not called
- `states:` = what the model reported about fridge/oven/tap/...; `[state]` rows come from a state change

In [ ]:
from datetime import datetime, timedelta
from modules.storage import EventStore
from query.llm_qa import fmt_row
from evaluation.log_accuracy import EpicVocab, load_truth, clip_name_parts, score_log, rows_from_store, print_report, print_details

VIDEO = CLIPS[0]
name = os.path.splitext(os.path.basename(VIDEO))[0]
store = EventStore(f'{WORK}/mnemo_log.db'); store.clear_activities()
START = datetime.now().replace(hour=9, minute=0, second=0, microsecond=0)   # pretend it was filmed at 9:00

sig, g = gate.signals(VIDEO)
scores = sig['hoi']
call = gate_decisions(scores)
print(f'TRIGGER: big model needed on {call.sum()} of {len(call)} windows (hands/objects/HOI-TCN took {g["gate_seconds"]}s)\n')
logger = ActionLogger(describer, store)
rows, stats = logger.run(VIDEO, start_time=START, call=call, scores=scores)
print('\n', stats)

print('\nSTORED LOG (this text is ALL that is kept, no video):')
for r in store.activities():
    print('  ', fmt_row(r))
print('\nWhat the big model actually wrote for the first described window:')
print([w for w in logger.windows if w['text']][0]['text'])
for obj, (st, ts) in logger.tracker.current().items():
    if st in ('open', 'on'):
        print(f'\n⚠  ALERT: the {obj} was last seen {st.upper()} at {ts:%H:%M:%S}')

vocab = EpicVocab(f'{EK}/EPIC_100_verb_classes.csv', f'{EK}/EPIC_100_noun_classes.csv')
vid, t0 = clip_name_parts(name)
truth = load_truth(ANN, vid, t0, stats['video_seconds'])
sc = score_log(rows_from_store(store, START), truth, vocab)
print()
print_report(sc, f'SCORE against the EPIC ground truth of {name}:')
print_details(sc)

### Ask questions (stage 2)
Edit the list. The answers use ONLY the log above.

In [ ]:
from query.llm_qa import answer
NOW = START + timedelta(hours=1)
QUESTIONS = [
    'What did I do in the kitchen this morning?',
    'Did I open the fridge?',
    'Did I close the fridge?',
    'Did I turn off the tap?',
    'Did I take my medicine?',
]
for q in QUESTIONS:
    r = answer(q, store, llm=describer, now=NOW)
    print(f'Q: {q}\nA: {r["answer"]}\n')

## 6. RESULTS 1: benchmark on all clips (~3-4 min per clip, about 1h15)
For every clip: one pass of the big model, then several systems are compared on the same model answers:

| System | What it is |
|---|---|
| `captions` | big model every 4 s: the usual "caption + LLM" memory (the baseline) |
| `+ states` | + object-state memory |
| `mnemo@0.5 … 0.8` | + our interaction trigger at different thresholds (fewer big-model calls) |
| `mnemo@T+everyK` | trigger + at least one look every K windows (hybrid: regular state checks) |
| `every-nth = mnemo@…` | the SAME number of calls, evenly spaced, no model (is our trigger better than a fixed rate?) |
| `direct VLM (no memory)` | (questions only) ask the big model directly about the whole clip |

Finished clips are cached, so if Kaggle stops, just run this cell again.

In [ ]:
from evaluation.benchmark import run_benchmark
from IPython.display import Markdown, display
RESULTS = f'{WORK}/results_{C.VLM_PROMPT_VERSION}_seed{SEED}'
results, summary = run_benchmark(CLIPS, describer, gate, ANN, EK, RESULTS)
display(Markdown(summary))

## 7. RESULTS 2: Experiment C, questions about a whole morning (~10 min)
The 20 clips become one morning (one clip every 7.5 minutes, about 2.5 hours in total). The morning's log is
also saved as `morning.db` (demo data for the app). Mnemo answers from its log;
the direct model gets 32 frames spread over all clips (it does not even see the idle gaps). Also asks
*"When did I …?"*, which needs a memory with times.

In [ ]:
from evaluation.long_qa import run_long_qa
long_res, when, long_md = run_long_qa(CLIPS, describer, ANN, EK, RESULTS, n_yes=48, n_no=24)
display(Markdown(long_md))

## 8. (Optional, ~1 h) Experiment A: trigger vs. fixed-rate on long 5-minute stretches
Coverage of ALL annotated actions vs. % of windows sent to the big model. Off by default; set `RUN_EXPERIMENT_A = True` to run it.

In [ ]:
RUN_EXPERIMENT_A = False
import json
if RUN_EXPERIMENT_A:
  vlm_s = numpy.mean([w['dt'] for f in glob.glob(f'{RESULTS}/vlm_cache/*.json') for w in json.load(open(f))['windows'] if w['text']])
  VLM_S, TP = f'{vlm_s:.2f}', ' '.join(TEST_P)
  print(f'measured big-model time per window: {vlm_s:.1f}s')
  !python -m tools.fetch_epic_windows --data "{EK}" --out "{WORK}/epic_cont" --select random --windows 10 --window-s 300 --workers 8 --participants {TP} --seed 0
  !python -m model.extract_features segments --videos "{WORK}/epic_cont" --annotations "{WORK}/epic_cont/segments.csv" --name epic_cont --out "{WORK}/features"
  !python -m evaluation.gating_experiment --features "{WORK}/features/epic_cont" --checkpoint models/hoi_tcn.pt --annotations "{ANN}" --vlm-seconds {VLM_S} --out "{RESULTS}/gating"
  from IPython.display import Image
  display(Image(f'{RESULTS}/gating/gating_curve.png'))

## 9. Save everything
Then download `mnemo_results.zip`: in the finished version's **Output** tab (or, in an open session: right panel →
**Output** → ↻ → `mnemo_results.zip` → ⋮ → **Download**).
It has the tables (`summary.md`, `long_qa.md`), every answer, and the cached model answers (for re-scoring on the laptop).

In [ ]:
shutil.copy(f'{WORK}/mnemo_log.db', RESULTS)
shutil.make_archive(f'{WORK}/mnemo_results', 'zip', RESULTS)
for f in ('summary.md', 'long_qa.md'):
    if os.path.exists(f'{RESULTS}/{f}'):
        print(open(f'{RESULTS}/{f}').read())
os.chdir(WORK)
from IPython.display import FileLink
display(FileLink('mnemo_results.zip'))